<a href="https://colab.research.google.com/github/akakdo999-stack/2026-DM/blob/main/discrete_math_graph_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 이산수학 08 그래프 실습 — DFS, BFS, 친구 추천, 감염 시뮬레이션

**작성자:** 김광수

**과제 구성**
1. graph 구조를 팀원과 팀원 친구의 이름으로 바꿔서 실습
2. DFS와 BFS 각각의 탐색 순서를 손으로 써 본 후 비교
3. (선택) networkx로 시각화하기
4. 감염 시뮬레이션으로 확장: 2단계 이내 전염 경로 추적 + 테스트 케이스(하/중/상)

## 1. 친구 관계 그래프 만들기

주어진 친구 목록은 다음과 같다.

| 사람 | 친구 |
|---|---|
| 김광수 | 파이리, 꼬북이, 이상해씨, 김민석, 김민혁 |
| 이윤건 | 박서준, 왕현식, 김민혁, 곽정훈, 박형준 |
| 김민석 | 최우진, 안우현, 이재용, 심문섭, 홍준서 |
| 김민혁 | 이윤건, 곽정훈, 최우진, 안우현, 김민준 |
| 곽정훈 | 이윤건, 김민혁, 최우진, 안우현, 배정우 |

친구 관계는 서로 친구인 관계이므로 **무방향 그래프**로 보고, 딕셔너리를 이용한 **인접 리스트**로 표현한다.
A의 목록에 B가 있으면 B의 목록에도 A를 넣었다. (예: 김민혁의 목록에는 김광수가 없지만, 김광수의 목록에 김민혁이 있으므로 김민혁 → 김광수도 추가)

손으로 쓴 탐색 순서와 맞추기 위해 각 사람의 친구 목록은 **가나다순**으로 정렬했다.

In [ ]:
# 1. 친구 관계 그래프 만들기
# 딕셔너리를 활용한 인접 리스트 (친구 목록은 가나다순)

graph = {
    '김광수': ['김민석', '김민혁', '꼬북이', '이상해씨', '파이리'],
    '파이리': ['김광수'],
    '꼬북이': ['김광수'],
    '이상해씨': ['김광수'],

    '김민석': ['김광수', '심문섭', '안우현', '이재용', '최우진', '홍준서'],
    '이재용': ['김민석'],
    '심문섭': ['김민석'],
    '홍준서': ['김민석'],

    '김민혁': ['곽정훈', '김광수', '김민준', '안우현', '이윤건', '최우진'],
    '김민준': ['김민혁'],

    '이윤건': ['곽정훈', '김민혁', '박서준', '박형준', '왕현식'],
    '박서준': ['이윤건'],
    '왕현식': ['이윤건'],
    '박형준': ['이윤건'],

    '곽정훈': ['김민혁', '배정우', '안우현', '이윤건', '최우진'],
    '배정우': ['곽정훈'],

    # 김민석, 김민혁, 곽정훈 세 사람 모두와 친구인 사람
    '최우진': ['곽정훈', '김민석', '김민혁'],
    '안우현': ['곽정훈', '김민석', '김민혁'],
}

# 친구 관계 출력
print("친구 관계 그래프")

for person in graph:
    print(person, ":", graph[person])

print("\n전체 정점 수:", len(graph))
print("전체 연결 수:", sum(len(friend) for friend in graph.values()) // 2)

In [ ]:
# 무방향 그래프 검사: A가 B의 친구면 B도 A의 친구여야 한다

def check_undirected(graph):
    for person, friends in graph.items():
        for friend in friends:
            if person not in graph.get(friend, []):
                print(f"오류: {person} → {friend} 는 있지만 {friend} → {person} 가 없음")
                return False
    return True

print("무방향 그래프인가?", check_undirected(graph))

## 2. DFS / BFS 탐색 (시작 정점: 김광수)

- **DFS(깊이 우선 탐색):** 한 경로를 끝까지 깊게 들어간 뒤 더 갈 곳이 없으면 되돌아간다. → 재귀 호출(스택) 사용
- **BFS(너비 우선 탐색):** 시작 정점에서 가까운 정점부터 단계별로 방문한다. → 큐 사용

In [ ]:
# DFS 탐색

def dfs(graph, v, visited, order):
    visited.add(v)
    order.append(v)
    print(v, end=' ')

    for neighbor in graph[v]:
        if neighbor not in visited:
            dfs(graph, neighbor, visited, order)

    return order


# BFS 탐색

from collections import deque

def bfs(graph, start):
    visited = set()
    queue = deque([start])
    order = []

    visited.add(start)

    while queue:
        v = queue.popleft()
        order.append(v)
        print(v, end=' ')

        for neighbor in graph[v]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return order


# 탐색 결과 출력

print("DFS 탐색 순서:")
dfs_order = dfs(graph, '김광수', set(), [])

print("\n\nBFS 탐색 순서:")
bfs_order = bfs(graph, '김광수')

### 손으로 쓴 탐색 순서와 비교

손으로 먼저 써 본 순서 (인접 정점은 가나다순으로 방문)

- **DFS:** 김광수 → 김민석 → 심문섭 → 안우현 → 곽정훈 → 김민혁 → 김민준 → 이윤건 → 박서준 → 박형준 → 왕현식 → 최우진 → 배정우 → 이재용 → 홍준서 → 꼬북이 → 이상해씨 → 파이리
- **BFS:** 김광수 → 김민석 → 김민혁 → 꼬북이 → 이상해씨 → 파이리 → 심문섭 → 안우현 → 이재용 → 최우진 → 홍준서 → 곽정훈 → 김민준 → 이윤건 → 배정우 → 박서준 → 박형준 → 왕현식

아래 코드로 손으로 쓴 결과와 프로그램 결과가 같은지 확인한다.

In [ ]:
hand_dfs = ['김광수', '김민석', '심문섭', '안우현', '곽정훈', '김민혁', '김민준', '이윤건', '박서준',
            '박형준', '왕현식', '최우진', '배정우', '이재용', '홍준서', '꼬북이', '이상해씨', '파이리']

hand_bfs = ['김광수', '김민석', '김민혁', '꼬북이', '이상해씨', '파이리', '심문섭', '안우현', '이재용',
            '최우진', '홍준서', '곽정훈', '김민준', '이윤건', '배정우', '박서준', '박형준', '왕현식']

print("DFS 손 풀이 == 코드 결과 :", hand_dfs == dfs_order)
print("BFS 손 풀이 == 코드 결과 :", hand_bfs == bfs_order)
print("방문한 정점 수 (DFS / BFS):", len(dfs_order), "/", len(bfs_order))

### DFS · BFS 비교 및 정리

| 구분 | DFS | BFS |
|---|---|---|
| 탐색법 | 한 경로를 깊게 탐색한 후 되돌아감 | 가까운 정점부터 단계별 탐색 |
| 자료구조 | 재귀 호출 / 스택 | 큐 |
| 이 그래프에서 | 김민석 → 심문섭 → 안우현 → 곽정훈 → 김민혁처럼 멀리 먼저 들어감 | 김광수의 친구 5명(거리 1)을 먼저 모두 방문한 뒤 거리 2, 3 순서로 방문 |

**요약** — DFS는 깊이 우선, BFS는 가까운 노드 우선이므로 방문 순서가 다르다.
이 그래프는 김민혁–이윤건–곽정훈 같은 **사이클**이 있어서 `visited`로 방문 여부를 확인하지 않으면 같은 정점을 여러 번 방문하게 된다.
또 모든 정점이 하나로 연결되어 있어서 김광수에서 한 번만 탐색해도 18명을 모두 방문한다.

## 3. 친구 추천 (BFS 응용: 2단계 거리 친구)

내 친구의 친구(거리 2)만 추천하고, 나와 직접 연결된 친구는 제외한다.

In [ ]:
# 2단계 친구 추천

def recommend_friend(graph, start):

    visited = {start}
    queue = deque([(start, 0)])   # (노드, 거리)
    recommended = []

    while queue:

        current, depth = queue.popleft()

        if depth == 2:
            recommended.append(current)
            continue

        for neighbor in graph[current]:

            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, depth + 1))

    print(f"\n{start}의 친구 추천 (2단계):")

    if recommended:
        for friend in recommended:
            print("-", friend)
    else:
        print("추천할 친구가 없습니다.")

    return recommended


# 친구 목록이 주어진 5명에 대해 친구 추천

for person in ['김광수', '이윤건', '김민석', '김민혁', '곽정훈']:
    recommend_friend(graph, person)

## 4. (선택) networkx로 시각화하기

코랩에서 한글이 깨지지 않도록 나눔 폰트를 먼저 설치한다.

In [ ]:
# 한글 폰트 설정 (코랩)
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if not os.path.exists(FONT_PATH):
    os.system('apt-get -qq install -y fonts-nanum > /dev/null')

if os.path.exists(FONT_PATH):
    fm.fontManager.addfont(FONT_PATH)
    FONT = fm.FontProperties(fname=FONT_PATH).get_name()
else:
    FONT = 'Noto Sans CJK JP'   # 코랩이 아닌 환경용 예비 폰트

plt.rc('font', family=FONT)
plt.rc('axes', unicode_minus=False)
print("사용 폰트:", FONT)

In [ ]:
import networkx as nx

G = nx.Graph()
for person, friends in graph.items():
    for friend in friends:
        G.add_edge(person, friend)

# 손으로 그린 그림과 비슷한 위치에 정점 배치
pos = {
    '파이리': (200, 50), '꼬북이': (340, 40), '이상해씨': (480, 50),
    '김광수': (340, 140),
    '이재용': (70, 210), '심문섭': (70, 290), '홍준서': (70, 370),
    '김민석': (190, 290), '김민혁': (500, 290), '김민준': (620, 210),
    '최우진': (340, 330), '안우현': (340, 450),
    '이윤건': (610, 430), '박서준': (630, 345), '왕현식': (635, 520), '박형준': (550, 610),
    '곽정훈': (470, 530), '배정우': (390, 610),
}
pos = {k: (x, -y) for k, (x, y) in pos.items()}   # 화면 좌표 → 그래프 좌표(위아래 뒤집기)

hubs = ['김광수', '이윤건', '김민석', '김민혁', '곽정훈']
colors = ['#F0997B' if n == '김광수' else '#AFA9EC' if n in hubs else '#D3D1C7' for n in G.nodes]

plt.figure(figsize=(10, 8))
nx.draw(G, pos, with_labels=True, node_color=colors, node_size=1800,
        font_family=FONT, font_size=11, edge_color='#888780')
plt.title(f"친구 관계 그래프 (정점 {G.number_of_nodes()}개, 간선 {G.number_of_edges()}개)", fontsize=14)
plt.show()

## 5. 감염 시뮬레이션: 2단계 이내 전염 경로 추적

**조건**
- 그래프는 인접 리스트 형태로 구성한다.
- 감염자는 1명이며, 이름으로 지정한다.
- `max_depth` 단계 이내로 연결된 노드들을 출력하되, **감염자 자신은 제외**한다.
- 감염 경로와 거리(레벨)를 함께 표시한다.

**방법** — BFS를 사용한다. BFS는 가까운 사람부터 방문하므로, 처음 방문할 때의 단계가 곧 **최단 거리(감염 단계)**가 된다.
큐에 `(사람, 단계, 경로)`를 함께 넣어서 경로를 추적한다.

In [ ]:
# BFS를 활용한 감염 시뮬레이션

def infection_simulation(graph, infected, max_depth=2, verbose=True):

    if infected not in graph:
        raise ValueError(f"존재하지 않는 사람입니다: {infected}")

    visited = {infected}
    queue = deque([(infected, 0, [infected])])   # (사람, 감염 단계, 감염 경로)

    infected_list = []   # 감염자 자신은 넣지 않는다

    while queue:

        current, depth, path = queue.popleft()

        if depth >= max_depth:      # 최대 단계에 도달하면 더 퍼뜨리지 않음
            continue

        for neighbor in graph[current]:

            if neighbor not in visited:

                visited.add(neighbor)

                next_depth = depth + 1
                next_path = path + [neighbor]

                infected_list.append((neighbor, next_depth, next_path))
                queue.append((neighbor, next_depth, next_path))

    if verbose:
        print("최초 감염자:", infected)
        print(f"\n감염 경로 추적 결과 ({max_depth}단계 이내, 총 {len(infected_list)}명)")

        for person, level, path in infected_list:
            print(f"{level}단계 감염: {person:<5}  경로: {' -> '.join(path)}")

    return infected_list


# 김광수를 최초 감염자로 지정

result = infection_simulation(graph, '김광수', 2)

In [ ]:
# 감염 단계별 색으로 시각화 (빨강: 최초 감염자, 주황: 1단계, 노랑: 2단계, 회색: 감염 안 됨)

level_of = {'김광수': 0}
for person, level, path in result:
    level_of[person] = level

level_color = {0: '#E24B4A', 1: '#EF9F27', 2: '#FAC775'}
colors = [level_color.get(level_of.get(n), '#D3D1C7') for n in G.nodes]

plt.figure(figsize=(10, 8))
nx.draw(G, pos, with_labels=True, node_color=colors, node_size=1800,
        font_family=FONT, font_size=11, edge_color='#888780')
plt.title("김광수 최초 감염 → 2단계 이내 감염자", fontsize=14)
plt.show()

## 6. 테스트 케이스

| 난이도 | 노드 수 | max depth |
|---|---|---|
| 하 | 10개 | 2 |
| 중 | 100개 | 10 |
| 상 | 1000개 | 50 |

테스트용 그래프는 `make_test_graph`로 만든다. (그래프 생성은 바이브 코딩으로 위임)
1. 최초 감염자 `P0`부터 `P1 - P2 - ... - P(max_depth)`까지 길이 `max_depth`인 **사슬**을 먼저 만들어서, 감염이 정확히 `max_depth` 단계까지 퍼지도록 한다.
2. 나머지 노드는 거리가 `max_depth`보다 작은 기존 노드에 무작위로 붙인다. → 모든 노드가 `max_depth` 단계 안에 들어온다.
3. 거리 차이가 1 이하인 노드끼리 무작위 간선을 더 추가해서 사이클이 있는 그래프로 만든다. (지름길이 생기지 않아서 가장 깊은 단계가 `max_depth`로 유지됨)

**검증 방법**
- 감염자 자신이 결과에 없는지
- 중복 감염자가 없는지
- 모든 경로가 실제 간선으로 이어져 있고, 경로 길이 = 단계인지
- 각 사람의 감염 단계가 networkx로 구한 최단 거리와 같은지
- 감염자 수가 (최단 거리가 `max_depth` 이하인 사람 수)와 같은지

In [ ]:
import random

def make_test_graph(n, max_depth, extra_edges=None, seed=0):
    """노드 n개, 감염자 P0 기준 최대 거리가 max_depth인 연결 그래프(인접 리스트)를 만든다."""
    rnd = random.Random(seed)
    g = {f'P{i}': [] for i in range(n)}
    dist = {'P0': 0}

    def add_edge(a, b):
        if a != b and b not in g[a]:
            g[a].append(b)
            g[b].append(a)

    # 1) 길이 max_depth 사슬
    for i in range(1, max_depth + 1):
        add_edge(f'P{i-1}', f'P{i}')
        dist[f'P{i}'] = i

    # 2) 나머지 노드는 거리 < max_depth 인 노드에 붙이기
    for i in range(max_depth + 1, n):
        parent = rnd.choice([v for v, d in dist.items() if d < max_depth])
        add_edge(parent, f'P{i}')
        dist[f'P{i}'] = dist[parent] + 1

    # 3) 무작위 간선 추가 (사이클 생성)
    if extra_edges is None:
        extra_edges = n // 5
    # 거리 차이가 1 이하인 두 노드만 연결 → 지름길이 생기지 않아 거리가 그대로 유지됨
    nodes = list(g)
    added = 0
    while added < extra_edges:
        a, b = rnd.choice(nodes), rnd.choice(nodes)
        if a != b and b not in g[a] and abs(dist[a] - dist[b]) <= 1:
            add_edge(a, b)
            added += 1

    return g


def check_result(g, start, max_depth, result):
    """감염 시뮬레이션 결과가 올바른지 검사한다."""
    G_test = nx.Graph()
    G_test.add_nodes_from(g)
    for a, friends in g.items():
        for b in friends:
            G_test.add_edge(a, b)
    true_dist = nx.single_source_shortest_path_length(G_test, start, cutoff=max_depth)

    names = [p for p, _, _ in result]
    assert start not in names, "감염자 자신이 결과에 포함됨"
    assert len(names) == len(set(names)), "중복 감염자 존재"
    for person, level, path in result:
        assert path[0] == start and path[-1] == person, "경로의 시작/끝이 잘못됨"
        assert len(path) - 1 == level, "경로 길이와 단계가 다름"
        assert all(path[i+1] in g[path[i]] for i in range(len(path) - 1)), "경로에 없는 간선이 있음"
        assert level == true_dist[person], f"{person}: 단계 {level} != 최단거리 {true_dist[person]}"
    assert len(result) == len(true_dist) - 1, "감염자 수가 최단거리 기준과 다름"
    return True


def run_test(title, n, max_depth, seed=0):
    g = make_test_graph(n, max_depth, seed=seed)
    edges = sum(len(v) for v in g.values()) // 2
    result = infection_simulation(g, 'P0', max_depth, verbose=(n <= 10))

    levels = {}
    for _, level, _ in result:
        levels[level] = levels.get(level, 0) + 1

    print(f"\n[{title}] 노드 {n}개, 간선 {edges}개, max depth {max_depth}")
    print(f"  감염자 수 (P0 제외): {len(result)}명")
    print(f"  가장 깊은 감염 단계: {max(levels)}")
    print("  단계별 감염자 수:", dict(sorted(levels.items())))
    deepest = max(result, key=lambda r: r[1])
    print(f"  가장 먼 감염 경로 예시 ({deepest[1]}단계): {' -> '.join(deepest[2][:6])}"
          + (" -> ... -> " + deepest[0] if len(deepest[2]) > 6 else ""))
    print("  검증 통과:", check_result(g, 'P0', max_depth, result) and max(levels) == max_depth)
    return g, result

In [ ]:
# 테스트 1 — 하: 노드 10개, max depth 2 (전체 경로 출력)
g_low, r_low = run_test("하", 10, 2)

In [ ]:
# 테스트 2 — 중: 노드 100개, max depth 10
g_mid, r_mid = run_test("중", 100, 10)

In [ ]:
# 테스트 3 — 상: 노드 1000개, max depth 50
g_high, r_high = run_test("상", 1000, 50)

In [ ]:
# 추가 테스트 — 예외 상황

# (1) 직접 만든 친구 그래프 검증
print("친구 그래프 2단계 결과 검증:", check_result(graph, '김광수', 2, result))

# (2) max_depth = 0 이면 아무도 감염되지 않음
print("max_depth=0 감염자 수:", len(infection_simulation(graph, '김광수', 0, verbose=False)))

# (3) 친구가 한 명뿐인 사람(파이리)에서 시작하면 1단계는 김광수 한 명
r = infection_simulation(graph, '파이리', 1, verbose=False)
print("파이리 1단계 감염자:", [p for p, _, _ in r])

# (4) 친구가 없는 사람만 있는 그래프
print("고립된 사람의 감염자 수:", len(infection_simulation({'홀로': []}, '홀로', 2, verbose=False)))

# (5) 존재하지 않는 이름
try:
    infection_simulation(graph, '피카츄', 2)
except ValueError as e:
    print("예외 발생:", e)

## 7. 바이브 코딩 정보

**사용 AI 서비스:** Claude (Anthropic)

**사용 프롬프트:**
> 이런 식을 구글 콜랩으로 과제를 해야 해.
> 김광수 = 파이리, 꼬북이, 이상해씨, 김민석, 김민혁 / 이윤건 = 박서준, 왕현식, 김민혁, 곽정훈, 박형준 / 김민석 = 최우진, 안우현, 이재용, 심문섭, 홍준서 / 김민혁 = 이윤건, 곽정훈, 최우진, 안우현, 김민준 / 곽정훈 = 이윤건, 김민혁, 최우진, 안우현, 배정우
> 여기 목록으로, 저기 코딩 참고용으로 해주고, 과제를 구글 콜랩 코딩해줘.
>
> (첨부: 과제 안내 이미지, 강의 자료 PDF, 참고 노트북)

**코드 설명 (코드 인터뷰 대비)**
- **그래프:** 딕셔너리 인접 리스트. 키는 사람, 값은 친구 리스트. 친구 관계는 양방향이라 무방향 그래프로 만들고 `check_undirected`로 확인했다.
- **DFS:** 재귀 호출로 구현. 방문한 정점을 `visited`(set)에 넣고, 아직 방문하지 않은 친구에게 재귀로 들어간다. 재귀 호출 자체가 스택 역할을 한다.
- **BFS:** `collections.deque`를 큐로 사용. 큐에 넣는 순간 `visited`에 추가해서 같은 사람이 큐에 두 번 들어가지 않게 했다.
- **친구 추천:** BFS에 거리(depth)를 같이 저장하고, 거리가 정확히 2인 사람만 추천한다.
- **감염 시뮬레이션:** BFS 큐에 `(사람, 단계, 경로)`를 저장한다. BFS는 가까운 사람부터 방문하므로 처음 방문했을 때의 단계가 최단 거리이고, 그때의 경로가 최단 감염 경로이다. `depth >= max_depth`이면 더 이상 퍼뜨리지 않는다. 감염자 자신은 `infected_list`에 넣지 않는다.
- **시간 복잡도:** 모든 정점과 간선을 한 번씩만 보므로 O(V + E). 1000개 노드도 바로 끝난다.
- **테스트:** 하/중/상 그래프를 직접 생성하고, networkx 최단 거리와 비교해서 결과가 맞는지 자동으로 검증했다.

## (첨부) 손으로 쓴 DFS · BFS 풀이 사진

In [ ]:
from google.colab import files
from IPython.display import display, Image

uploaded = files.upload()

for filename in uploaded:
    display(Image(data=uploaded[filename]))